# 01 — Chuẩn bị dữ liệu YOLO

Convert COCO JSON → YOLO format cho Card Detector và Field Detector.

### Chuẩn bị trước khi chạy

**Bước 1:** Zip folder data trên máy local:
```
Nén thành cccd_data.zip, bên trong gồm:
  data/
    cccd.v1i.coco/
      train/   valid/   test/   (toàn bộ ảnh)
    processed/
      splits/
        train.json   val.json   test.json
```

**Bước 2:** Upload `cccd_data.zip` lên Google Drive → chuột phải → "Get link" → chọn "Anyone with the link" → copy **File ID** từ link.

Ví dụ link: `https://drive.google.com/file/d/`**`1AbCdEfGhIjKlMnOpQrStUvWx`**`/view`
→ File ID là phần in đậm.

**Bước 3:** Dán File ID vào cell bên dưới.

## 0. Cài đặt

In [1]:
!pip install gdown pyyaml -q

## 1. Download & Giải nén data

In [2]:
import gdown

FILE_ID = "1CzgcY8qbT0HkcBpJdHB5-5rNXvO9kSlZ"

gdown.download(id=FILE_ID, output="/content/cccd_data.zip", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1CzgcY8qbT0HkcBpJdHB5-5rNXvO9kSlZ
From (redirected): https://drive.google.com/uc?id=1CzgcY8qbT0HkcBpJdHB5-5rNXvO9kSlZ&confirm=t&uuid=cfb4a5e1-cebc-4f60-9139-a58319b9bfaf
To: /content/cccd_data.zip
100%|██████████| 629M/629M [00:10<00:00, 59.2MB/s]


'/content/cccd_data.zip'

In [3]:
import zipfile

with zipfile.ZipFile("/content/cccd_data.zip", "r") as z:
    z.extractall("/content/")

!ls /content/data/

Giai nen cccd_data.zip...
cccd.v1i.coco  cccd.v1i.coco.zip  interim  processed  yolo


## 2. Setup

In [4]:
import json
import shutil
from collections import defaultdict
from pathlib import Path

import yaml
from tqdm.notebook import tqdm

BASE_DIR   = Path("/content")
RAW_DIR    = BASE_DIR / "data/cccd.v1i.coco"
SPLITS_DIR = BASE_DIR / "data/processed/splits"
YOLO_DIR   = BASE_DIR / "data/yolo"

CARD_CLASSES  = ["card"]
FIELD_CLASSES = ["id", "name", "birth", "origin", "address", "title"]

assert RAW_DIR.exists(),    f"Khong tim thay: {RAW_DIR}"
assert SPLITS_DIR.exists(), f"Khong tim thay: {SPLITS_DIR}"
YOLO_DIR.mkdir(parents=True, exist_ok=True)


## 3. Build bảng tra cứu ảnh

In [5]:
fname_to_path: dict[str, Path] = {}
for split in ["train", "valid", "test"]:
    for img_path in (RAW_DIR / split).glob("*.jpg"):
        fname_to_path[img_path.name] = img_path

## 4. Hàm convert COCO → YOLO

In [6]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = max(0.0, min(1.0, (x + w / 2) / img_w))
    cy = max(0.0, min(1.0, (y + h / 2) / img_h))
    w  = max(0.0, min(1.0, w / img_w))
    h  = max(0.0, min(1.0, h / img_h))
    return cx, cy, w, h


def convert_split(coco_path, split_name, target_classes, out_dir, fname_to_path):
    img_out   = out_dir / "images" / split_name
    label_out = out_dir / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)

    with open(coco_path, encoding="utf-8") as f:
        coco = json.load(f)

    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    class_to_idx   = {cls: i for i, cls in enumerate(target_classes)}

    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        if cat_id_to_name.get(ann["category_id"]) in class_to_idx:
            anns_by_img[ann["image_id"]].append(ann)

    stats = {"copied": 0, "skipped": 0}
    for img_info in tqdm(coco["images"], desc=f"  {split_name}", leave=False):
        src = fname_to_path.get(img_info["file_name"])
        if src is None or not src.exists():
            stats["skipped"] += 1
            continue

        dst = img_out / img_info["file_name"]
        if not dst.exists():
            shutil.copy2(src, dst)

        lines = []
        for ann in anns_by_img.get(img_info["id"], []):
            cls = cat_id_to_name[ann["category_id"]]
            cx, cy, w, h = coco_bbox_to_yolo(ann["bbox"], img_info["width"], img_info["height"])
            lines.append(f"{class_to_idx[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        (label_out / (Path(img_info["file_name"]).stem + ".txt")).write_text(
            "\n".join(lines), encoding="utf-8"
        )
        stats["copied"] += 1

    return stats

## 5. Convert Card Detector

In [7]:
card_dir = YOLO_DIR / "card"
print("=== Card Detector ===")
for split_name, fname in [("train", "train.json"), ("val", "val.json"), ("test", "test.json")]:
    stats = convert_split(SPLITS_DIR / fname, split_name, CARD_CLASSES, card_dir, fname_to_path)
    print(f"  {split_name}: {stats['copied']} anh, bo qua {stats['skipped']}")

card_yaml = {
    "path": str(card_dir),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(CARD_CLASSES),
    "names": CARD_CLASSES,
}
(card_dir / "data.yaml").write_text(yaml.dump(card_yaml, allow_unicode=True))
print(f"Saved: {card_dir / 'data.yaml'}")

=== Card Detector ===


  train:   0%|          | 0/3074 [00:00<?, ?it/s]

  train: 3074 anh, bo qua 0


  val:   0%|          | 0/699 [00:00<?, ?it/s]

  val: 699 anh, bo qua 0


  test:   0%|          | 0/626 [00:00<?, ?it/s]

  test: 626 anh, bo qua 0
Saved: /content/data/yolo/card/data.yaml


## 6. Convert Field Detector

In [8]:
field_dir = YOLO_DIR / "field"
print("=== Field Detector ===")
for split_name, fname in [("train", "train.json"), ("val", "val.json"), ("test", "test.json")]:
    stats = convert_split(SPLITS_DIR / fname, split_name, FIELD_CLASSES, field_dir, fname_to_path)
    print(f"  {split_name}: {stats['copied']} anh, bo qua {stats['skipped']}")

field_yaml = {
    "path": str(field_dir),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(FIELD_CLASSES),
    "names": FIELD_CLASSES,
}
(field_dir / "data.yaml").write_text(yaml.dump(field_yaml, allow_unicode=True))
print(f"Saved: {field_dir / 'data.yaml'}")

=== Field Detector ===


  train:   0%|          | 0/3074 [00:00<?, ?it/s]

  train: 3074 anh, bo qua 0


  val:   0%|          | 0/699 [00:00<?, ?it/s]

  val: 699 anh, bo qua 0


  test:   0%|          | 0/626 [00:00<?, ?it/s]

  test: 626 anh, bo qua 0
Saved: /content/data/yolo/field/data.yaml


## 7. Verify

In [9]:
for detector, out_dir in [("card", card_dir), ("field", field_dir)]:
    print(f"\n[{detector}]")
    for split in ["train", "val", "test"]:
        n_imgs   = len(list((out_dir / "images" / split).glob("*.jpg")))
        n_labels = len(list((out_dir / "labels" / split).glob("*.txt")))
        status   = "OK" if n_imgs == n_labels else "MISMATCH"
        print(f"  {split:5s}: {n_imgs} images | {n_labels} labels [{status}]")


[card]
  train: 3074 images | 3074 labels [OK]
  val  : 699 images | 699 labels [OK]
  test : 626 images | 626 labels [OK]

[field]
  train: 3074 images | 3074 labels [OK]
  val  : 699 images | 699 labels [OK]
  test : 626 images | 626 labels [OK]
